# 04 · Sweep analysis and a first SINDy fit

This notebook has three parts:

1. **Sweep overview.** Condense all `5m_attack_*` datasets into one table and a few plots:
   win rates, battle durations in seconds, and which settings give repeated (deterministic) games.
2. **Level A for one dataset.** Look at the team-level trajectories of a chosen dataset,
   split by who won.
3. **First SINDy fit.** Fit a model of how team health evolves, evaluate it by simulating
   held-out episodes, and save the result as a run in `runs/`.

Settings that you may want to change are collected in the next cell.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysindy as ps

from sindy_wm.data import storage
from sindy_wm.evaluation.plots import TEAM_COLORS, plot_many_episodes
from sindy_wm.evaluation.sweep import episodes_by_time, heatmap, load_all_indexes, summarize
from sindy_wm.models.sindy import (
    evaluate,
    fit_sindy,
    plot_predictions,
    save_run,
    summarize_eval,
    to_trajectory,
)
from sindy_wm.paths import PROJECT_ROOT, RAW_DIR
from sindy_wm.states.aggregate import level_a

plt.rcParams["figure.dpi"] = 100

# --- Part 1: Which experiment to analyse
EXPERIMENT = "exp03_10m_random_aggression_stepmul"
EXPERIMENT_DIR = RAW_DIR / EXPERIMENT

# --- Part 2 and 3: the dataset to analyse and fit (pick it after looking at Part 1)
DATASET = "10m_attack_a070_s1_v1"

# --- Part 3: modelling choices
STATE_COLUMNS = [
    "health_blue",
    "health_red",
    "alive_blue",
    "alive_red",
]  # Level A variables used as SINDy state
TRIM_TO_ENGAGEMENT = True  # start each trajectory when the first damage is dealt
DOWNSAMPLE = 2  # keep every k-th step (2 at step_mul 1 gives 1/8 s per point)
SMOOTH_WINDOW = 9  # Savitzky-Golay window for derivatives, in (downsampled) points
TEST_FRACTION = 0.2  # share of episodes held out for evaluation
SPLIT_SEED = 0  # seed for the train/test split

RUNS_DIR = PROJECT_ROOT / "runs"

## Part 1 · Sweep overview

### 1.1 One table for all episodes

Each dataset's manifest records its parameters, and its episode index records the outcomes.
`load_all_indexes` combines them into one table with one row per episode.
Durations are converted to **seconds**, because a step means something different at each `step_mul`.

In [ ]:
episodes = load_all_indexes(EXPERIMENT_DIR)
print(f"{episodes['dataset'].nunique()} datasets, {len(episodes)} episodes")
episodes.head()

### 1.2 Summary per dataset

For each dataset:

- **win rate** with a 95% interval. With 100 episodes the interval is roughly ±10 percentage points
  near 50%, so differences smaller than that are not meaningful.
- **duration** in seconds.
- **distinct outcomes**: how many different (length, final health) combinations occur.
  A value of 1 means every episode is the same game repeated.

In [ ]:
summary = (
    episodes.groupby(["dataset", "step_mul", "aggression"])
    .apply(summarize, include_groups=False)
    .reset_index()
    .sort_values(["step_mul", "aggression"], ignore_index=True)
)
summary.style.format(
    {
        "win_rate": "{:.0%}",
        "win_low": "{:.0%}",
        "win_high": "{:.0%}",
        "duration_mean_s": "{:.1f}",
        "duration_std_s": "{:.1f}",
        "steps_mean": "{:.1f}",
        "truncated": "{:.0%}",
        "n": "{:.0f}",
        "distinct_outcomes": "{:.0f}",
    }
)

### 1.3 Win rate across the sweep

Each cell shows the Blue win rate and, below it, its 95% interval.
Cells marked *repeated* contain essentially one game played many times, so their
0% or 100% is a single outcome rather than a rate.

In [ ]:
fig, ax, win_table = heatmap(summary, "win_rate", "Blue win rate")
by_cell = summary.set_index(["aggression", "step_mul"])
for i, a in enumerate(win_table.index):
    for j, s in enumerate(win_table.columns):
        row = by_cell.loc[(a, s)]
        label = f"{row.win_rate:.0%}\n[{row.win_low:.0%}, {row.win_high:.0%}]"
        if row.distinct_outcomes <= 2:
            label += "\nrepeated"
        dark = row.win_rate < 0.2 or row.win_rate > 0.8
        ax.text(j, i, label, ha="center", va="center", fontsize=8, color="white" if dark else "black")
plt.show()

**What to look for:** win rate should rise with aggression and fall with `step_mul`.
A plausible reason: with probability 1 − aggression Blue takes a random action, and that
action lasts one decision. At `step_mul` 1 it lasts 1/16 s; at `step_mul` 8 it lasts 0.5 s.

### 1.4 Battle duration in seconds

If the duration in seconds is similar at every `step_mul`, the battle itself is the same
physical event; `step_mul` then only changes how finely it is recorded and how often Blue decides.

In [ ]:
step_muls = sorted(summary["step_mul"].unique())
position = {s: k for k, s in enumerate(step_muls)}  # evenly spaced x positions

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for n, (a, group) in enumerate(summary.groupby("aggression")):
    x = group["step_mul"].map(position) + (n - 1.5) * 0.06  # small offset so error bars don't overlap
    axes[0].errorbar(
        x,
        group["duration_mean_s"],
        yerr=group["duration_std_s"],
        marker="o",
        capsize=3,
        label=f"aggression {a}",
    )
    axes[1].plot(x, group["steps_mean"], marker="o", label=f"aggression {a}")
axes[0].set(ylabel="duration (s)", title="Battle duration, mean ± std", ylim=(0, None))
axes[1].set(ylabel="steps per episode", title="Data points per episode", ylim=(0, None))
for ax in axes:
    ax.set_xticks(range(len(step_muls)), step_muls)
    ax.set_xlabel("step_mul")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 1.5 Candidate datasets for SINDy

A good first dataset has **balanced outcomes** (both teams win sometimes, so the model sees both
regimes), **varied episodes** (not one game repeated) and **many points per episode**.

In [ ]:
candidates = summary[summary["win_rate"].between(0.3, 0.7) & (summary["distinct_outcomes"] > 10)].sort_values(
    "steps_mean", ascending=False
)
candidates[["dataset", "win_rate", "steps_mean", "duration_mean_s", "distinct_outcomes"]]

Set `DATASET` in the first cell to the dataset you want to use in Parts 2 and 3,
and rerun from there.

## Part 2 · Level A trajectories of one dataset

### 2.1 Load and compute Level A

Level A is computed per episode with `normalize=True`, so health and alive counts run from 1 down to 0.
The result has a two-level index, `(episode_idx, step)`. Time in seconds is added as a column.

In [ ]:
DATASET = "10m_attack_a100_s1_v1"

dataset_dir = EXPERIMENT_DIR / DATASET
manifest = storage.load_manifest(dataset_dir)
index = storage.load_episode_index(dataset_dir).set_index("episode_idx")
dt = manifest["config"]["env"]["seconds_per_step"]

snaps = storage.load_snapshots(dataset_dir)
states = snaps.groupby("episode_idx").apply(level_a, normalize=True, include_groups=False)
states["t"] = states.index.get_level_values("step") * dt

print(f"{DATASET}: {len(index)} episodes, dt = {dt} s, Blue win rate {index['battle_won'].mean():.0%}")
states.head()

### 2.2 Won and lost episodes side by side

Each line is one episode, plotted against time in seconds.

In [ ]:
won_ids = index.index[index["battle_won"]]
lost_ids = index.index[~index["battle_won"]]

for ids, label in [(won_ids, "Blue won"), (lost_ids, "Blue lost")]:
    if len(ids) == 0:
        print(f"No episodes where {label.lower()}")
        continue
    fig = plot_many_episodes(episodes_by_time(states, ids), title=f"{DATASET}, {label}", alpha=0.2)
    fig.axes[-1].set_xlabel("time (s)")
    plt.show()

### 2.3 Phase portrait: health of Blue against health of Red

This view removes time and shows how the two teams wear each other down.
Lanchester's **square law** (each side's losses proportional to the other side's strength)
predicts curves along which $a\,B^2 - b\,R^2$ stays constant: hyperbola-like curves that end on
one of the axes. Under the **linear law** the curves would be straight lines.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for i in index.index:
    ep = states.loc[i]
    color = TEAM_COLORS["blue"] if index.loc[i, "battle_won"] else TEAM_COLORS["red"]
    ax.plot(ep["health_red"], ep["health_blue"], color=color, alpha=0.25, linewidth=1)
ax.plot([], [], color=TEAM_COLORS["blue"], label="Blue won")
ax.plot([], [], color=TEAM_COLORS["red"], label="Blue lost")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="equal health")
ax.set(
    xlabel="health_red (normalized)",
    ylabel="health_blue (normalized)",
    xlim=(0, 1.02),
    ylim=(0, 1.02),
    aspect="equal",
    title=DATASET,
)
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.show()

## Part 3 · A first SINDy fit

### 3.1 From Level A to trajectories

Choices made here (all set in the first cell):

- **State:** `STATE_COLUMNS`, by default the two team healths. The alive counts are left out because
  they are step functions, which suit derivative-based fitting poorly.
- **One trajectory per episode.** Episodes are never stacked into one array, since the jump between
  the end of one episode and the start of the next would look like a huge derivative.
- **Trim to engagement:** before the first shot, units are only walking towards each other and health
  does not change. A model that only sees health cannot tell this phase apart, so trajectories start
  one step before the first damage.
- **Downsample** to reduce the staircase of individual shots; derivatives are smoothed further with
  a Savitzky–Golay filter (`SMOOTH_WINDOW`).
- **Split by episode** into training and test episodes.

In [ ]:
trajectories = {
    i: to_trajectory(states.loc[i], STATE_COLUMNS, TRIM_TO_ENGAGEMENT, DOWNSAMPLE) for i in index.index
}

rng = np.random.default_rng(SPLIT_SEED)
episode_ids = rng.permutation(index.index.to_numpy())
n_test = int(round(TEST_FRACTION * len(episode_ids)))
test_ids = sorted(episode_ids[:n_test].tolist())
train_ids = sorted(episode_ids[n_test:].tolist())

dt_fit = dt * DOWNSAMPLE
lengths = pd.Series({i: len(trajectories[i]) for i in index.index})
print(f"{len(train_ids)} training and {len(test_ids)} test episodes, dt = {dt_fit} s")
print(f"Points per trajectory: mean {lengths.mean():.0f}, min {lengths.min()}, max {lengths.max()}")
if lengths.min() < SMOOTH_WINDOW:
    print("Warning: some trajectories are shorter than SMOOTH_WINDOW; lower it or DOWNSAMPLE.")

### 3.2 Fit

Feature library: polynomials of the state. Degree 1 can express Lanchester-type laws
($\dot B = -a R$, $\dot R = -b B$); degree 2 adds products and squares.
The optimizer (STLSQ) sets coefficients below `threshold` to zero, which keeps the model sparse.

In [ ]:
x = [trajectories[i][STATE_COLUMNS].to_numpy() for i in train_ids]

In [ ]:
x[0]

In [ ]:
model = fit_sindy(trajectories, train_ids, STATE_COLUMNS, dt_fit, SMOOTH_WINDOW, degree=1, threshold=0.02)
model.print()

**What to look for:** under the square law, `health_blue'` depends mainly on `health_red` with a
negative coefficient, and the other way round. A constant term means losses that do not depend on
the opponent's strength, and a term in a team's own health means its losses depend on itself.

### 3.3 Evaluate by simulating test episodes

Fitting derivatives well is not the same as predicting a battle well. For each test episode, the
model is integrated forward and compared with what actually happened, in two ways:

- **From the start of the engagement.** Every episode starts from (almost) the same state, so a
  deterministic model predicts (almost) the same battle every time: it can only learn the *average*
  fight. Which side wins depends on random events during the battle, so the winner accuracy from
  the start cannot be expected to beat always guessing the more common outcome (`majority_baseline`).
  The RMSE shows how well the model captures the average course of a battle.
- **From midway.** Starting from the actual state halfway through the trajectory, where episodes
  differ, tests whether the model knows how the current balance of health decides the rest of the fight.
  Its naive baseline is `leader_baseline_midway`: predicting that whoever has more health midway wins.
  The model is only useful here if it beats that.

The predicted winner is the team with more health left at the time the real battle ended.

In [ ]:
summarize_eval(model, test_ids, trajectories, STATE_COLUMNS, index["battle_won"])

### 3.4 Compare a few model choices

A small grid over library degree and sparsity threshold. Compare winner accuracies against their baselines,
and prefer the **simplest model that predicts about as well as the best one**: fewer terms are easier to interpret and less likely to overfit.
The comparison uses the test episodes; for the final thesis results, keep a separate set of episodes
(for example a dataset with new seeds) that is never used for choosing models.

In [ ]:
comparison = []
for degree in [1, 2]:
    for threshold in [0.005, 0.02, 0.05, 0.1]:
        m = fit_sindy(trajectories, train_ids, STATE_COLUMNS, dt_fit, SMOOTH_WINDOW, degree, threshold)
        metrics_m = summarize_eval(m, test_ids, trajectories, STATE_COLUMNS, index["battle_won"])
        comparison.append({"degree": degree, "threshold": threshold, **metrics_m})
comparison = pd.DataFrame(comparison)
comparison

Set the chosen model below. Then look at a few test episodes: actual (solid), simulated from the start (dashed)
and from midway (dotted).

In [ ]:
DEGREE, THRESHOLD = 1, 0.1  # chosen from the comparison above

model = fit_sindy(trajectories, train_ids, STATE_COLUMNS, dt_fit, SMOOTH_WINDOW, DEGREE, THRESHOLD)
model.print()
metrics = summarize_eval(model, test_ids, trajectories, STATE_COLUMNS, index["battle_won"])
per_episode = evaluate(model, test_ids, trajectories, STATE_COLUMNS, index["battle_won"])
metrics

In [ ]:
test_won = [i for i in test_ids if index.loc[i, "battle_won"]][:2]
test_lost = [i for i in test_ids if not index.loc[i, "battle_won"]][:2]
fig_predictions = plot_predictions(
    model,
    test_won + test_lost,
    trajectories,
    STATE_COLUMNS,
    index["battle_won"],
    title=f"{DATASET}: test episodes",
)
plt.show()

An existing run folder is never overwritten. `save_run` lives in `sindy_wm.models.sindy`
(moved there once the format settled; see the README's Next steps history).

In [ ]:
run_config = {
    "description": "First SINDy fit on Level A team health",
    "dataset": {"name": DATASET, "experiment": EXPERIMENT, "created": manifest["created"]},
    "split": {
        "train": [int(i) for i in train_ids],
        "test": [int(i) for i in test_ids],
        "seed": SPLIT_SEED,
        "test_fraction": TEST_FRACTION,
    },
    "preprocessing": {
        "state_columns": STATE_COLUMNS,
        "normalize": True,
        "trim_to_engagement": TRIM_TO_ENGAGEMENT,
        "downsample": DOWNSAMPLE,
        "dt": dt_fit,
    },
    "model": {
        "library": "PolynomialLibrary",
        "degree": DEGREE,
        "optimizer": "STLSQ",
        "threshold": THRESHOLD,
        "differentiation": "SmoothedFiniteDifference (Savitzky-Golay)",
        "smooth_window": SMOOTH_WINDOW,
        "pysindy": ps.__version__,
    },
}

run_dir = save_run(
    runs_dir=RUNS_DIR,
    name=f"levelA_health_poly{DEGREE}",
    config=run_config,
    model=model,
    state_columns=STATE_COLUMNS,
    metrics=metrics,
    per_episode=per_episode,
    figures={"test_predictions.png": fig_predictions},
)
print(f"Saved run to {run_dir}")
print(sorted(p.relative_to(run_dir).as_posix() for p in run_dir.rglob("*")))

## Part 4 · How much do SMOOTH_WINDOW and DOWNSAMPLE matter?

SINDy fits the model's coefficients against the *estimated* derivative, not the true one — so if
that estimate is dominated by the discreteness of individual hits (health changes in chunks, not
continuously: see "About modelling" in the README) rather than the underlying rate, the fitted
equations should be noticeably sensitive to exactly how much smoothing/downsampling is applied.
If the fit is instead picking up the real dynamics, the equations should stay close to each other
as these are varied moderately.

Health actually changes close to every 2 raw steps on average here (5 units, each on its own
cooldown), so the chosen `SMOOTH_WINDOW=9` at `DOWNSAMPLE=2` spans roughly 18 raw steps — about 9
hit events, comfortably more than one. The sweep below checks that directly rather than taking it
on faith. Both sweeps refit on the same `train_ids`/`test_ids` split as above, at `DEGREE`/
`THRESHOLD` fixed to whatever was chosen in 3.4, so only the preprocessing changes.

In [ ]:
smooth_sweep = []
for window in [5, 9, 13, 17]:
    m = fit_sindy(trajectories, train_ids, STATE_COLUMNS, dt_fit, window, DEGREE, THRESHOLD)
    metrics_w = summarize_eval(m, test_ids, trajectories, STATE_COLUMNS, index["battle_won"])
    smooth_sweep.append({"smooth_window": window, **metrics_w, "equations": m.equations(precision=3)})
smooth_sweep = pd.DataFrame(smooth_sweep)

for _, row in smooth_sweep.iterrows():
    print(f"window={row.smooth_window:>2}:")
    for s, eq in zip(STATE_COLUMNS, row.equations, strict=True):
        print(f"  ({s})' = {eq}")

smooth_sweep[["smooth_window", "rmse_midway", "winner_acc_midway", "n_terms"]]

### Downsample, with the smoothing window scaled to match

Changing `DOWNSAMPLE` changes `dt` too, so `SMOOTH_WINDOW` is rescaled with it here to keep the
window spanning roughly the same number of *raw* steps (~18, matching `DOWNSAMPLE=2`,
`SMOOTH_WINDOW=9`) at every factor — otherwise a bigger `DOWNSAMPLE` with the same
`SMOOTH_WINDOW` would span far fewer hit events, confounding the two.

In [ ]:
downsample_sweep = []
raw_span = SMOOTH_WINDOW * DOWNSAMPLE  # raw steps the chosen settings cover
for downsample in [1, 2, 4]:
    window = max(5, round(raw_span / downsample / 2) * 2 + 1)  # nearest odd, at least 5
    traj_d = {
        i: to_trajectory(states.loc[i], STATE_COLUMNS, TRIM_TO_ENGAGEMENT, downsample) for i in index.index
    }
    shortest = min(len(t) for t in traj_d.values())
    if shortest <= window:
        print(f"downsample={downsample}: skipped, shortest trajectory {shortest} <= window {window}")
        continue
    dt_d = dt * downsample
    m = fit_sindy(traj_d, train_ids, STATE_COLUMNS, dt_d, window, DEGREE, THRESHOLD)
    metrics_d = summarize_eval(m, test_ids, traj_d, STATE_COLUMNS, index["battle_won"])
    downsample_sweep.append(
        {
            "downsample": downsample,
            "smooth_window": window,
            **metrics_d,
            "equations": m.equations(precision=3),
        }
    )
downsample_sweep = pd.DataFrame(downsample_sweep)

for _, row in downsample_sweep.iterrows():
    span = row.smooth_window * row.downsample
    print(f"downsample={row.downsample}, window={row.smooth_window} (raw span {span}):")
    for s, eq in zip(STATE_COLUMNS, row.equations, strict=True):
        print(f"  ({s})' = {eq}")

downsample_sweep[["downsample", "smooth_window", "rmse_midway", "winner_acc_midway", "n_terms"]]

**What to look for:** if the dominant coefficients (the `health_red` term in `(health_blue)'`, the
`health_blue` term in `(health_red)'`) stay close across both tables, the fit is picking up the
real dynamics rather than being driven by how finely the staircase was smoothed — the current
`SMOOTH_WINDOW`/`DOWNSAMPLE` choice is then just a convenience, not a hidden assumption the result
depends on. Large swings instead would mean the chosen settings are still in the "messy
derivative" regime, and the equations in 3.2–3.4 shouldn't be trusted without picking a window
that stabilizes first.

## Part 5 · Slider: watch a bad derivative break the model

Part 4 measured sensitivity to *how much* smoothing/downsampling is applied — on this dataset,
both turned out to barely matter. That's reassuring, but it doesn't really *show* a bad
derivative estimate breaking anything, because pooling across 400 training trajectories is
already fairly robust to those choices. To make the failure mode this notebook has been warning
about actually visible, the slider below injects Gaussian noise directly into the **training**
trajectories' health values before fitting (test episodes are always left clean, so what's
plotted is always "model fit on noisy data vs. the real, unmodified battle").

Push the noise level up and watch:

- the fitted equations drift, then the `health_red`/`health_blue` interaction term disappears
  entirely once its estimated coefficient drops below `STLSQ`'s `threshold` — the model collapses
  to two decoupled, physically wrong equations (each team losing health on its own, independent
  of the other's strength)
- `winner_acc_midway` falls back toward the naive baseline
- the simulated-vs-actual plot visibly stops tracking the real battle

`continuous_update=False` means it only refits when you release the slider, not on every pixel of
drag.

In [ ]:
import ipywidgets as widgets


def noisy_training_trajectories(noise_level, seed):
    # Corrupts only the training trajectories; test entries stay the original, clean objects
    nrng = np.random.default_rng(seed)
    out = dict(trajectories)
    for i in train_ids:
        noisy = trajectories[i].copy()
        noisy[STATE_COLUMNS] = noisy[STATE_COLUMNS] + nrng.normal(0, noise_level, noisy[STATE_COLUMNS].shape)
        out[i] = noisy
    return out


def show_noisy_fit(noise_level):
    traj_n = noisy_training_trajectories(noise_level, seed=round(noise_level * 1000))
    m = fit_sindy(traj_n, train_ids, STATE_COLUMNS, dt_fit, SMOOTH_WINDOW, DEGREE, THRESHOLD)
    # clean test data -- only the training fit is corrupted
    metrics_n = summarize_eval(m, test_ids, trajectories, STATE_COLUMNS, index["battle_won"])

    print(f"noise level: {noise_level:.2f}  (0.00 = the real, unmodified training data)")
    for s, eq in zip(STATE_COLUMNS, m.equations(precision=3), strict=True):
        print(f"  ({s})' = {eq}")
    print(
        f"  winner_acc_midway={metrics_n['winner_acc_midway']:.2f}"
        f" (leader_baseline_midway={metrics_n['leader_baseline_midway']:.2f})"
        f"   rmse_midway={metrics_n['rmse_midway']:.3f}   n_terms={metrics_n['n_terms']}"
    )

    plot_predictions(
        m,
        test_won + test_lost,
        trajectories,
        STATE_COLUMNS,
        index["battle_won"],
        title=f"noise={noise_level:.2f}",
    )
    plt.show()


widgets.interact(
    show_noisy_fit,
    noise_level=widgets.FloatSlider(
        value=0.0,
        min=0.0,
        max=0.3,
        step=0.01,
        description="noise",
        readout_format=".2f",
        continuous_update=False,
    ),
)